# Adım 5: Özellik Mühendisliği (Feature Engineering)
**Proje:** Spotify Büyük Veri Analizi  
**Veri Seti:** Spotify Tracks Dataset (~114K şarkı)  
Bu notebook'ta ML modeli için anlamlı 5 yeni özellik üretip Delta Lake Gold katmanına kaydedildi.

#### 1.Adım - Kütüphaneleri ve Spark Oturumunu Başlatma İşlemi Yapıldı

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, min as spark_min, max as spark_max, when, udf, avg, count
)
from pyspark.sql.types import StringType, IntegerType, DoubleType
import warnings
warnings.filterwarnings('ignore')  

# spark oturumu oluşturma delta lake ile
spark = SparkSession.builder \
    .appName("Spotify-FeatureEngineering") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

# Log seviyesini WARN'a çekiyoruz - konsolu temiz tutar
#spark.sparkContext.setLogLevel("WARN")

print("Spark oturumu başarıyla başladı...")
print(f"Spark versiyonu: {spark.version}")

Spark oturumu başarıyla başladı...
Spark versiyonu: 3.5.0


#### 2.Adım Veriyi Delta Lake Silver Dosyasından Yükleme İşlemi

In [3]:
# deltalake silver dosyasından temizlenmiş verinin okunması ve kontrolu
SILVER_PATH = "/home/jovyan/delta-lake/silver"
GOLD_PATH   = "/home/jovyan/delta-lake/gold/features"

df = spark.read.format("delta").load(SILVER_PATH)
#databriks test için 
#df = spark.read.parquet("/Volumes/workspace/default/silver/")
print(f"Veri yüklendi...")
print(f"Toplam satır: {df.count():,}")
print(f"Toplam kolon: {len(df.columns)}")

Veri yüklendi...
Toplam satır: 89,740
Toplam kolon: 23


#### 3. Adım - özellik 1- energy_danceability_ratio
EDA Adım 9'da türe göre enerji ve dans edilebilirlik karşılaştırıldığında bazı türlerin yüksek enerjili ama düşük dans edilebilirliğe sahip olduğu gözlemlendi.Bu özellik o ayrımı sayısal olarak ifade etmek için üretildi.

In [4]:
# enerji/danceability : Değer yüksekse şarkı enerjik ama dans edilemez → Rock, metal gibi 'agresif/sert' müzik türlerini diğerlerinden ayırt etmek için kullanılır.
# Paydaya 0.0001 ekledik ki danceability 0 gelirse hata vermesin

df = df.withColumn("energy_danceability_ratio",col("energy")/(col("danceability") + 0.0001))

# hesaplanan degerlerin kontrolu yapildi mean/min/max seklinde 
stats = df.select(
    spark_min("energy_danceability_ratio").alias("minimum"),
    spark_max("energy_danceability_ratio").alias("maksimum"),
    avg("energy_danceability_ratio").alias("ortalama")
).collect()[0]

print("energy_danceability_ratio")
print(f"  Minimum  : {stats['minimum']:.4f}")
print(f"  Maksimum : {stats['maksimum']:.4f}")
print(f"  Ortalama : {stats['ortalama']:.4f}")

print("en yüksek energy_danceability_ratio")
df.orderBy(col("energy_danceability_ratio").desc()) \
  .select("track_name", "artists", "track_genre",
          "energy", "danceability", "energy_danceability_ratio") \
  .show(10, truncate=30)

print("en dusuk energy_danceability_ratio")
df.orderBy(col("energy_danceability_ratio").asc()) \
  .select("track_name", "artists", "track_genre",
          "energy", "danceability", "energy_danceability_ratio") \
  .show(10, truncate=30)

energy_danceability_ratio
  Minimum  : 0.0000
  Maksimum : 9990.0001
  Ortalama : 3.4189
en yüksek energy_danceability_ratio
+------------------------------+------------------------------+-----------+------+------------+-------------------------+
|                    track_name|                       artists|track_genre|energy|danceability|energy_danceability_ratio|
+------------------------------+------------------------------+-----------+------+------------+-------------------------+
|Hotel Hair Dryer - Non-Stat...|Deep Sleep Hair Dryers;Hair...|      sleep| 0.999|         0.0|        9990.000128746033|
|               Continuous Rain|Rain Sounds;Sounds Of Natur...|      sleep| 0.998|         0.0|        9980.000257492065|
|               Continuous Rain|Rain for Deep Sleep;Yoga;Th...|      sleep| 0.998|         0.0|        9980.000257492065|
|             Ankara Rain Skies|                Sound Sleeping|      sleep| 0.994|         0.0|        9940.000176429749|
|        The Early Mo

Çıkarım: En yüksek skorlar sleep türünden geliyor. Bu şarkılarda energy yüksek (beyaz gürültü, yağmur sesi gibi sürekli ses var) ama danceability sıfır. En düşük skorlarda ise energy neredeyse sıfır olan sessiz guitar, goth ve emo parçaları var. Özellik beklenen davranışı gösteriyor. Rock/metal yerine sleep öne çıkmasının nedeni Spotify'ın energy'yi agresiflik değil ses yoğunluğu olarak ölçmesi olabilir.

#### 4. Adım - ozellik 2- loudness_normalized
EDA Adım 3.1'de loudness'ın -49.5 ile 4.5 arasında değiştiği gözlemlenmişti.Negatif değerler ML modeline doğrudan verilmesi doğru olmadığı için 0-1 arasına normalize edilme işlemi yapıldı

In [5]:
# once ses yuksekliği normalizasyon islemi için  min ve max değerleri hesaplandı 
min_loudness = df.select(spark_min("loudness")).collect()[0][0]
max_loudness = df.select(spark_max("loudness")).collect()[0][0]

print(f"Loudness min: {min_loudness:.2f} dB")
print(f"Loudness max: {max_loudness:.2f} dB")

# normalize edilme islemi yapildi
df = df.withColumn(
    "loudness_normalized",
    (col("loudness") - min_loudness) / (max_loudness - min_loudness)
)

stats = df.select(
    spark_min("loudness_normalized").alias("minimum"),
    spark_max("loudness_normalized").alias("maksimum"),
    avg("loudness_normalized").alias("ortalama")
).collect()[0]

print("loudness normalize değerleri")
print(f"  Minimum  : {stats['minimum']:.4f}")
print(f"  Maksimum : {stats['maksimum']:.4f}")
print(f"  Ortalama : {stats['ortalama']:.4f}")

# orjinal ve yeni normalize degerlerin karsilastirmasi
print("\nOrijinal vs Normalize karşılaştırma (5 satır):")
df.select("track_name", "loudness", "loudness_normalized").show(5, truncate=25)

Loudness min: -49.53 dB
Loudness max: 4.53 dB
loudness normalize değerleri
  Minimum  : 0.0000
  Maksimum : 1.0000
  Ortalama : 0.7590

Orijinal vs Normalize karşılaştırma (5 satır):
+-------------------------+--------+-------------------+
|               track_name|loudness|loudness_normalized|
+-------------------------+--------+-------------------+
|                   Better|  -6.644| 0.7932782037483795|
|Find Me - Sigma VIP Remix|  -2.544| 0.8691156576885397|
|       Sandwiches de Miga|  -8.103| 0.7662911760250471|
|              Mister Love| -10.258| 0.7264302590464271|
|528Hz Energía curativa...| -36.082|0.24876529089665067|
+-------------------------+--------+-------------------+
only showing top 5 rows



Çıkarım: loudness değişkeni -49.53 ile 4.53 dB arasında değişmektedir. Negatif değerlerin ML modeline doğrudan verilmesi uygun olmadığından min-max normalizasyonu uygulanarak 0-1 arasına ölçeklendirilmiştir. Normalize sonrası ortalama 0.7590 olup şarkıların büyük çoğunluğunun yüksek ses seviyesine sahip olduğu görülmektedir. Bu özellik tür tahmini modelinde ses yüksekliğini anlamlı bir girdi olarak sunmaktadır.

#### 5. adım - ozellik 3- tempo_category
EDA Adım 12'de yavas/orta/hizli gruplarının şarkı sayısı ve popülerlik dağılımı incelenmişti.tempo sayısal değerini kategoriye çeviriyoru
Sürekli bir sayı yerine anlamlı gruplar oluşturmak bazı modellerin örüntüleri daha iyi öğrenmesini saglayabilir. Orijinal tempo kolonuna ek olarak tempo sayısal degerini 3 kategori olarak gruplanması planlandı mle daha yorumlanabilir bilgi olarak sunmak icin

In [6]:
# tempo sayısal değerini 3 kategoriye çeviriyoruz
df = df.withColumn(
    "tempo_category",
    when(col("tempo") < 100, "yavas")       # 100 BPM'den yavaş
    .when((col("tempo") >= 100) & (col("tempo") <= 140), "orta")  # 100-140 BPM arası
    .otherwise("hizli")                     # 140 BPM'den hızlı
)

print("tempo_category")
# her kategoride kaç şarkı ve ortalama popülerlik var
df.groupBy("tempo_category") \
  .agg(
      count("*").alias("sarki_sayisi"),
      avg("tempo").alias("ort_tempo"),
      avg("popularity").alias("ort_populerlik")
  ) \
  .orderBy("ort_tempo") \
  .show()

print("Tempo ve kategori (30 örnek):")
df.select("track_name", "track_genre", "tempo", "tempo_category").show(30, truncate=30)

tempo_category
+--------------+------------+------------------+------------------+
|tempo_category|sarki_sayisi|         ort_tempo|    ort_populerlik|
+--------------+------------+------------------+------------------+
|         yavas|       23488| 85.50079007215331| 33.01881811989101|
|          orta|       43100|120.95551330194672|33.000533642691416|
|         hizli|       23152|161.19867761528894| 33.75051831375259|
+--------------+------------+------------------+------------------+

Tempo ve kategori (30 örnek):
+------------------------------+-----------------+-------+--------------+
|                    track_name|      track_genre|  tempo|tempo_category|
+------------------------------+-----------------+-------+--------------+
|                        Better|            chill|143.064|         hizli|
|     Find Me - Sigma VIP Remix|    drum-and-bass|174.986|         hizli|
|            Sandwiches de Miga|        punk-rock| 88.419|         yavas|
|                   Mister Love|  

Çıkarım:tempo_category dağılımına bakıldığında şarkıların yarısından fazlası orta tempo grubunda (100-140 BPM) yer alıyor. Üç grubun ortalama popülerliği birbirine çok yakın (33.0-33.7) olup tempo'nun tek başına popülerliği belirlemediği görülüyor. Tür bazında incelendiğinde aynı türün farklı tempo kategorilerinde çıkabildiği gözlemlendi; örneğin black-metal orta, grunge hızlı, study ise yavaş kategorisinde yer aldı. Bu durum tempo_category'nin tek başına tür tahmini için yeterli olmadığını ancak loudness_normalized ve energy_acoustic_contrast gibi diğer özelliklerle birlikte modele anlamlı katkı sağlayabileceğini göstermektedir.

#### 6.Adım - ozellik 4 -energy_acoustic_contrast
EDA Adım 7 korelasyon heatmap'te acousticness ile energy arasında -0.72 korelasyon gözlemlenmişti. Pozitif değer akustik baskın türleri (classical, piano, folk) temsil ederken negatif değer yüksek enerjili sert türleri (metal, grindcore, rock) temsil ediyor. Tür tahminine doğrudan katkı sağlayan güçlü bir ayırt edici özellik olarak ml modeline verilebilir.

In [7]:
# energy_acoustic_contrast = acousticness - energy
df = df.withColumn(
    "energy_acoustic_contrast",
    col("acousticness") - col("energy")
)

stats = df.select(
    spark_min("energy_acoustic_contrast").alias("minimum"),
    spark_max("energy_acoustic_contrast").alias("maksimum"),
    avg("energy_acoustic_contrast").alias("ortalama")
).collect()[0]

print("energy_acoustic_contrast")
print(f"  Minimum  : {stats['minimum']:.4f}")
print(f"  Maksimum : {stats['maksimum']:.4f}")
print(f"  Ortalama : {stats['ortalama']:.4f}")

# doğrulama
print("\nEn yüksek skorlar (akustik baskın):")
df.orderBy(col("energy_acoustic_contrast").desc()) \
  .select("track_name", "track_genre", "acousticness", "energy", "energy_acoustic_contrast") \
  .show(5, truncate=30)

print("\nEn düşük skorlar (sert/yoğun türler):")
df.orderBy(col("energy_acoustic_contrast").asc()) \
  .select("track_name", "track_genre", "acousticness", "energy", "energy_acoustic_contrast") \
  .show(5, truncate=30)

energy_acoustic_contrast
  Minimum  : -0.9994
  Maksimum : 0.9959
  Ortalama : -0.3062

En yüksek skorlar (akustik baskın):
+------------------------------+-----------+------------+-------+------------------------+
|                    track_name|track_genre|acousticness| energy|energy_acoustic_contrast|
+------------------------------+-----------+------------+-------+------------------------+
|Deeper Sounding Vacuum Clea...|      sleep|       0.996| 5.9E-5|                0.995941|
|Deeper Sounding Vacuum Clea...|      sleep|       0.996| 5.9E-5|                0.995941|
|                       Konbini|      piano|       0.996|5.44E-4|                0.995456|
|          Céntrate En Ti Mismo|world-music|       0.996|0.00147|              0.99452996|
|Miroirs, M. 43: V. La vallé...|  classical|       0.995|7.56E-4|                0.994244|
+------------------------------+-----------+------------+-------+------------------------+
only showing top 5 rows


En düşük skorlar (sert/yoğun tü

energy_acoustic_contrast değerleri -0.9994 ile 0.9959 arasında değişmektedir. En yüksek skorlarda sleep, piano ve classical türleri öne çıkarken en düşük skorlarda black-metal, death-metal ve grindcore yer aldı. Ortalama -0.3062 olup veri setinde yüksek enerjili şarkıların daha fazla olduğu görülmektedir. Bu özellik akustik ve sert türler arasındaki farkı başarıyla yakalamakta olup tür tahmini modelinde güçlü bir ayırt edici özellik olarak kullanılabilir.

#### 7. Adım- ozellik 5- dancefloor_score
EDA Adım 10 Radar chart analizinde EDM türünün danceability ve energy eksenlerinde yüksek, acousticness ekseninde ise çok düşük değerler aldığı gözlemlenmişti. Classical tam tersi profil sergilerken hip-hop orta energy ile öne çıkıyordu. Bu üç özelliği tek bir çarpımsal skora indirgemek EDM ve dance türlerini diğerlerinden ayırt etmek için güçlü bir özellik olarak kullanılabilir

In [8]:
df = df.withColumn(
    "dancefloor_score",
    col("danceability") * col("energy") * (1 - col("acousticness"))
)

stats = df.select(
    spark_min("dancefloor_score").alias("minimum"),
    spark_max("dancefloor_score").alias("maksimum"),
    avg("dancefloor_score").alias("ortalama")
).collect()[0]

print("dancefloor_score")
print(f"  Minimum  : {stats['minimum']:.4f}")
print(f"  Maksimum : {stats['maksimum']:.4f}")
print(f"  Ortalama : {stats['ortalama']:.4f}")

# doğrulama
print("\nEn yüksek dancefloor_score (dans pisti türleri):")
df.orderBy(col("dancefloor_score").desc()) \
  .select("track_name", "track_genre", "danceability", "energy", "acousticness", "dancefloor_score") \
  .show(10, truncate=30)

print("\nEn düşük dancefloor_score:")
df.orderBy(col("dancefloor_score").asc()) \
  .select("track_name", "track_genre", "danceability", "energy", "acousticness", "dancefloor_score") \
  .show(10, truncate=30)

dancefloor_score
  Minimum  : 0.0000
  Maksimum : 0.9564
  Ortalama : 0.2803

En yüksek dancefloor_score (dans pisti türleri):
+------------------------------+--------------+------------+------+------------+----------------+
|                    track_name|   track_genre|danceability|energy|acousticness|dancefloor_score|
+------------------------------+--------------+------------+------+------------+----------------+
|                   Most Wanted| chicago-house|       0.976|  0.98|     8.27E-5|      0.95640093|
|The Music In Me - Original Mix| chicago-house|        0.97| 0.952|      0.0323|       0.8936129|
|                 Addams Groove|          kids|        0.97|  0.92|     0.00669|      0.88642985|
|                 Addams Groove|          kids|        0.97|  0.92|      0.0067|      0.88642097|
|Nobody Likes the Records Th...|         happy|       0.903|  0.98|     9.77E-4|      0.88407546|
|I'm Gonna Dance Tonight - O...| chicago-house|       0.973| 0.895|     7.11E-4|       0.

dancefloor_score değerleri 0.0 ile 0.956 arasında değişmektedir. En yüksek skorlarda chicago-house, disco ve detroit-techno türleri öne çıkmakta olup bu türlerin yüksek danceability ve energy, düşük acousticness değerlerine sahip olduğu görülmektedir. En düşük skorlarda ise danceability değeri sıfır olan sleep türü yer almaktadır; danceability sıfır olduğunda çarpımsal formül gereği skor da sıfır çıkmaktadır. Özellik dans pisti türlerini diğerlerinden başarıyla ayırt etmekte olup tür tahmini modelinde EDM ve dans türlerini tanımlamada güçlü bir özellik olarak kullanılabilir.

####  özelliklerin veriseti üzerinde kontrol edilmesi

In [12]:
yeni_ozellikler = [
    "energy_danceability_ratio",
    "loudness_normalized",
    "tempo_category",
    "energy_acoustic_contrast",
    "dancefloor_score",
]

print("Yeni Özellikler (15 örnek):")
display(df.select(["track_name", "track_genre"] + yeni_ozellikler).limit(15).toPandas())

sayisal_yeni = [o for o in yeni_ozellikler if o != "tempo_category"]
print("\nİstatistikler:")
display(df.select(sayisal_yeni).describe().toPandas())

print("\nTempo Kategori Dağılımı:")
display(df.groupBy("tempo_category").count().orderBy("tempo_category").toPandas())

Yeni Özellikler (15 örnek):


,track_name,track_genre,energy_danceability_ratio,loudness_normalized,tempo_category,energy_acoustic_contrast,dancefloor_score
0,Better,chill,0.768227,0.793278,hizli,-0.155000,0.197487
1,Find Me - Sigma VIP Remix,drum-and-bass,2.139244,0.869116,hizli,-0.882900,0.366641
2,Sandwiches de Miga,punk-rock,1.981684,0.766291,yavas,-0.775990,0.305226
3,Mister Love,honky-tonk,0.335371,0.726430,orta,0.611000,0.023893
4,528Hz Energía curativa profunda,world-music,0.058199,0.248765,hizli,0.985960,0.000004
5,Yemyeşil Bir Deniz,j-pop,1.168484,0.746019,orta,0.084000,0.096065
6,It's a Prison Workout,comedy,1.603145,0.819507,yavas,-0.153000,0.117977
7,Park Bench,study,0.624767,0.770287,yavas,0.247000,0.101229
8,Reach Out,alt-rock,1.070728,0.813514,orta,-0.702550,0.467223
9,Terminal Odyssey,black-metal,5.268210,0.798568,orta,-0.932998,0.165141



İstatistikler:


,summary,energy_danceability_ratio,loudness_normalized,energy_acoustic_contrast,dancefloor_score
0,count,89740,89740,89740,89740
1,mean,3.4189033598584255,0.7589664976861054,-0.3061735018714145,0.28026318103068615
2,stddev,116.26634000994305,0.09658209491822747,0.5545303379874025,0.19618664457009052
3,min,0.0,0.0,-0.99935,0.0
4,max,9990.000128746033,1.0,0.995941,0.95640093



Tempo Kategori Dağılımı:


,tempo_category,count
0,hizli,23152
1,orta,43100
2,yavas,23488


##### ozellik tablosunu deltalake gold dosyasına kaydetme islemi 

In [18]:
# ml için golda kaydedilecek attributelar
gold_kolonlar = [
    # benzersiz ozellikler
    "track_id","track_name", "artists", "album_name", 
    #hedef
    "track_genre","popularity",
    # orijinal sayısal veriler
    "danceability", "energy", "loudness", "tempo",
    "speechiness", "acousticness", "instrumentalness", "liveness",
    "valence", "duration_ms", "key", "mode", "time_signature",
    # orijinal kategorik ozellik
    "explicit",
    # kafka timestamp
    "kafka_timestamp",'user_id', 'event_type',
    # uretilen +5 ozellik
    "energy_danceability_ratio",  "loudness_normalized", "tempo_category","energy_acoustic_contrast","dancefloor_score",           
]
mevcut_kolonlar = [c for c in gold_kolonlar if c in df.columns]
df_gold = df.select(mevcut_kolonlar)
print(f"df'deki toplam kolon sayısı   : {len(df.columns)}")
print(f"Gold'a yazılacak kolon sayısı : {len(mevcut_kolonlar)}")
print(f"Gold'a yazılacak satır sayısı : {df_gold.count():,}")
print(f"\nEksik kolonlar (df'de yok)    : {[c for c in gold_kolonlar if c not in df.columns]}")

df'deki toplam kolon sayısı   : 28
Gold'a yazılacak kolon sayısı : 28
Gold'a yazılacak satır sayısı : 89,740

Eksik kolonlar (df'de yok)    : []


In [19]:
#yazdırma islemi
GOLD_PATH = "/home/jovyan/delta-lake/gold/features"

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_PATH)

print(f"Özellik tablosu Gold katmanına kaydedildi")
print(f"Konum: {GOLD_PATH}")

Özellik tablosu Gold katmanına kaydedildi
Konum: /home/jovyan/delta-lake/gold/features


In [27]:
# Kaydın başarılı olduğunu dogrulama işlemi
df_verify = spark.read.format("delta").load(GOLD_PATH)
print("Gold Katmanından Okunan Veri")
print(f"Satır sayısı   : {df_verify.count():,}")
print(f"Kolon sayısı   : {len(df_verify.columns)}")
print("\nKolon listesi:")
for kolon in df_verify.columns:
    print(f"{kolon}")
print("\nİlk 5 satır (yeni özellikler):")
display(df_verify.select(
    "track_genre",
    "energy_danceability_ratio",
    "loudness_normalized",
    "tempo_category",
    "energy_acoustic_contrast",
    "dancefloor_score"
).limit(5).toPandas())
print("\nİlk 5 satır:")

display(df_verify.limit(5).toPandas())

Gold Katmanından Okunan Veri
Satır sayısı   : 89,740
Kolon sayısı   : 28

Kolon listesi:
track_id
track_name
artists
album_name
track_genre
popularity
danceability
energy
loudness
tempo
speechiness
acousticness
instrumentalness
liveness
valence
duration_ms
key
mode
time_signature
explicit
kafka_timestamp
user_id
event_type
energy_danceability_ratio
loudness_normalized
tempo_category
energy_acoustic_contrast
dancefloor_score

İlk 5 satır (yeni özellikler):


,track_genre,energy_danceability_ratio,loudness_normalized,tempo_category,energy_acoustic_contrast,dancefloor_score
0,comedy,1.454952,0.741394,hizli,0.01200,0.086961
1,industrial,1.170589,0.717681,orta,-0.74270,0.485598
2,hard-rock,0.882832,0.728076,orta,-0.59564,0.402155
3,drum-and-bass,1.874348,0.833398,hizli,-0.89688,0.429660
4,indian,1.100811,0.837671,yavas,-0.05800,0.157852



İlk 5 satır:


,track_id,track_name,artists,album_name,track_genre,popularity,danceability,energy,loudness,tempo,...,time_signature,explicit,kafka_timestamp,user_id,event_type,energy_danceability_ratio,loudness_normalized,tempo_category,energy_acoustic_contrast,dancefloor_score
0,0017XiMkqbTfF2AUOzlhj6,Thanksgiving Chicken,Chad Daniels,Busy Being Awesome,comedy,24,0.536,0.780,-9.449,173.912003,...,3,1,2026-05-12T11:00:41.468316,e54e1dbf-e815-4233-b377-7048681cace1,track_played,1.454952,0.741394,hizli,0.01200,0.086961
1,0051nJ5xbRu8kuqPTYa9l7,"Hips, Tits, Lips, Power!",Pigface,The Best Of Pigface,industrial,20,0.650,0.761,-10.731,110.067001,...,4,0,2026-05-12T11:02:37.335168,9f7d0330-a977-4f37-b303-9c11c0eaddd5,track_played,1.170589,0.717681,orta,-0.74270,0.485598
2,006Bi4j0yzwOc3y69GOlYV,Computadores Fazem Arte,Chico Science;Nação Zumbi,Da Lama Ao Caos,hard-rock,36,0.675,0.596,-10.169,119.902000,...,4,0,2026-05-12T11:02:05.250766,7a03f497-ed9e-4ee8-9c16-5803b36bb8ea,track_played,0.882832,0.728076,orta,-0.59564,0.402155
3,006c9li2Mybyg5vm6doEfO,Finish Line,Logistics;Zara Kershaw,Electric Sun,drum-and-bass,16,0.479,0.898,-4.475,174.001999,...,4,0,2026-05-12T11:01:09.059132,20f743fa-2840-47e4-a842-61b1c74da762,track_played,1.874348,0.833398,hizli,-0.89688,0.429660
4,0072MKNpMeJNx8aRfTNfQW,Shaayraana,Pritam;Arijit Singh,Holiday - A Soldier Is Never Off Duty (Origina...,indian,58,0.604,0.665,-4.244,93.962997,...,4,0,2026-05-12T11:02:25.693903,104d0ff1-484a-46d8-8f9c-8b04be1d2ff1,track_played,1.100811,0.837671,yavas,-0.05800,0.157852
